In [1]:
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset, Dataset
import pandas as pd
import torch
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

C:\Users\FA007\.conda\envs\new2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
device

device(type='cuda')

In [4]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import random
from collections import defaultdict
from genomic_benchmarks.dataset_getters.pytorch_datasets import get_dataset
# Import the genomic benchmark dataset from Hugging Face
from genomic_benchmarks.dataset_getters.pytorch_datasets import DemoMouseEnhancers

# --------------------
# Data Preprocessing
# --------------------




    # Load the genomic benchmark dataset from Hugging Face
train_dset = get_dataset("human_nontata_promoters", split="train", version=0)
test_dset  = get_dataset("human_nontata_promoters", split="test", version=0)

    # Extract sequences and labels from the datasets
train_seqs = [item[0] for item in train_dset]
train_labels = [item[1] for item in train_dset]
test_seqs = [item[0] for item in test_dset]
test_labels = [item[1] for item in test_dset]
import pandas as pd
# converting the lists to train and test csv
train_df = pd.DataFrame({
    'sequence':train_seqs,
    'target': train_labels
})
test_df = pd.DataFrame({
    'sequence':test_seqs,
    'target': test_labels
})   



In [5]:

test_df_new = pd.read_csv(r"D:\genomic benchmark\human_tata promoters\dnabert-2\newtest.csv")

In [6]:
def kmer_tokenizer(seq, k=6):
    return " ".join([seq[i:i+k] for i in range(len(seq)-k+1)])

train_df["kmer"] = train_df["sequence"].apply(lambda x: kmer_tokenizer(x))
test_df["kmer"] = test_df["sequence"].apply(lambda x: kmer_tokenizer(x))

In [7]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df[["kmer", "target"]])
test_dataset = Dataset.from_pandas(test_df[["kmer", "target"]])

In [8]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Use AutoTokenizer to handle correct tokenizer class
tokenizer = AutoTokenizer.from_pretrained("zhihan1996/DNABERT-2-117M", trust_remote_code=True)

# Load DNABERT-2 model for classification (binary classification: num_labels=2)
model = AutoModelForSequenceClassification.from_pretrained("zhihan1996/DNABERT-2-117M", num_labels=2, trust_remote_code=True).to(device)


C:\Users\FA007\.cache\huggingface\modules\transformers_modules\zhihan1996\DNABERT-2-117M\d064dece8a8b41d9fb8729fbe3435278786931f1\bert_layers.py:126: UserWarning: Unable to import Triton; defaulting MosaicBERT attention implementation to pytorch (this will reduce throughput when using this model).
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at zhihan1996/DNABERT-2-117M and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [9]:
def tokenize_function(examples):
    return tokenizer(
        examples["kmer"],
        padding="max_length",       # pad all sequences to max_length
        truncation=True,
        max_length=512              # ensure fixed 512 tokens
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)


# 6) Now rename "target" → "labels"
train_dataset = train_dataset.rename_column("target", "labels")
test_dataset  = test_dataset.rename_column("target", "labels")
train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

Map: 100%|██████████| 9034/9034 [00:01<00:00, 5598.77 examples/s]


In [10]:
# train_dataset = train_dataset.rename_column("target", "labels")
# test_dataset  = test_dataset.rename_column("target", "labels")
# train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
# test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

In [11]:
from transformers import TrainingArguments

def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds),
        "recall": recall_score(labels, preds),
        "f1": f1_score(labels, preds)
    }

# Define training arguments
training_args = TrainingArguments(
    output_dir="D:\genomic benchmark\datasets\dnabert",

    disable_tqdm=False,
    # evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    evaluation_strategy="no", 
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
)

C:\Users\FA007\.conda\envs\new2\Lib\site-packages\transformers\training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [12]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
   
    compute_metrics=compute_metrics,
)


In [13]:
import time

# Start timer
start_time = time.time()

# Train the model
trainer.train()

# End timer
end_time = time.time()
elapsed = end_time - start_time

# Report time
print(f"\n🕒 Training took {elapsed:.2f} seconds ({elapsed/60:.2f} minutes)")


Step,Training Loss
10,0.600600
20,0.511200
30,0.498300
40,0.602100
50,0.455700
60,0.543800
70,0.451400
80,0.416800
90,0.525300
100,0.469900



🕒 Training took 7495.82 seconds (124.93 minutes)


In [14]:
model.save_pretrained("D:\genomic benchmark\datasets\models and tokenizer/dnabert2-finetuned_humannontata")


In [15]:
tokenizer.save_pretrained("D:\genomic benchmark\datasets\models and tokenizer/dnabert2-finetuned_humannontata")


('D:\\genomic benchmark\\datasets\\models and tokenizer/dnabert2-finetuned_humannontata\\tokenizer_config.json',
 'D:\\genomic benchmark\\datasets\\models and tokenizer/dnabert2-finetuned_humannontata\\special_tokens_map.json',
 'D:\\genomic benchmark\\datasets\\models and tokenizer/dnabert2-finetuned_humannontata\\tokenizer.json')

In [16]:
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset, Dataset
import pandas as pd
import torch
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

In [18]:
test_df

,sequence,target,kmer
0,AGAGCAGAAGACCGAAAGGTGAGTCGGCCTGCGGACTCTTCCGGCC...,0,AGAGCA GAGCAG AGCAGA GCAGAA CAGAAG AGAAGA GAAG...
1,ACTCTCGTTTGGGGTAGGGTCTTCCGGCTTTTTGTCGGGGGGACCC...,0,ACTCTC CTCTCG TCTCGT CTCGTT TCGTTT CGTTTG GTTT...
2,ATGGGTTTATCAGCAGCAAGCGGGCGGGAGAGGGACGCAGGCGGAC...,0,ATGGGT TGGGTT GGGTTT GGTTTA GTTTAT TTTATC TTAT...
3,GATGGGTTTATCAGCAGCAAGCGGGCGGGAGAGGGACGCAGGCGGA...,0,GATGGG ATGGGT TGGGTT GGGTTT GGTTTA GTTTAT TTTA...
4,ACACTGTTGGGATTTGAAAGGCATTCATATGTTTCCTTGTCCAGAA...,0,ACACTG CACTGT ACTGTT CTGTTG TGTTGG GTTGGG TTGG...
...,...,...,...
9029,CTCTGTTACAGCTGGTTTGCACTGGGATTAGTTGCTCTTTGCATGA...,1,CTCTGT TCTGTT CTGTTA TGTTAC GTTACA TTACAG TACA...
9030,TTTCATCTGGTCTCTTCTCCCCCGCTTCCCACCTCCCAATTCCCGC...,1,TTTCAT TTCATC TCATCT CATCTG ATCTGG TCTGGT CTGG...
9031,AGGATGCGGACCCGGGACGCCCCCGTGAGCTCACTGCGCCTGGCTG...,1,AGGATG GGATGC GATGCG ATGCGG TGCGGA GCGGAC CGGA...
9032,AGGAGGTTTTAGAGCTGGGCCAGATGTGTACCCTTCCCGGCTTTCT...,1,AGGAGG GGAGGT GAGGTT AGGTTT GGTTTT GTTTTA TTTT...


In [19]:
import pandas as pd
test_df_im = pd.read_csv(r"D:\genomic benchmark\human_tata promoters\non_tata_vs_non_promoter_external_new.csv")


In [20]:
def kmer_tokenizer(seq, k=6):
    return " ".join([seq[i:i+k] for i in range(len(seq)-k+1)])


test_df["kmer"] = test_df["sequence"].apply(lambda x: kmer_tokenizer(x))
test_df_im["kmer"] = test_df_im["sequence"].apply(lambda x: kmer_tokenizer(x))

In [21]:
test_dataset = Dataset.from_pandas(test_df[["kmer", "target"]])
test_dataset_im = Dataset.from_pandas(test_df_im[["kmer", "target"]])

In [22]:
def tokenize_function(examples):
    return tokenizer(
        examples["kmer"],
        padding="max_length",       # pad all sequences to max_length
        truncation=True,
        max_length=512              # ensure fixed 512 tokens
    )


test_dataset = test_dataset.map(tokenize_function, batched=True)
test_dataset_im = test_dataset_im.map(tokenize_function, batched=True)

# 6) Now rename "target" → "labels"

test_dataset  = test_dataset.rename_column("target", "labels")
test_dataset_im  = test_dataset_im.rename_column("target", "labels")
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
test_dataset_im.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

Map: 100%|██████████| 47542/47542 [00:08<00:00, 5416.66 examples/s]


In [23]:
from torch.utils.data import DataLoader

test_dataloader = DataLoader(
    test_dataset, batch_size=16, shuffle=False
)


In [24]:
import torch
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

model.eval()
model.to("cuda" if torch.cuda.is_available() else "cpu")

all_preds = []
all_labels = []
i=0
import time

# Start timer
start_time = time.time()
with torch.no_grad():
    for batch in test_dataloader:
        input_ids = batch['input_ids'].to(model.device)
        attention_mask = batch['attention_mask'].to(model.device)
        labels = batch['labels'].to(model.device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        preds = torch.argmax(logits, dim=-1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        i+=1
        if i%100==0:
            print(str(i)+" done")
end_time = time.time()
elapsed = end_time - start_time

# Report time
print(f"\n🕒 Evaluation took {elapsed:.2f} seconds ({elapsed/60:.2f} minutes)")


100 done
200 done
300 done
400 done
500 done

🕒 Evaluation took 259.26 seconds (4.32 minutes)


In [25]:
accuracy = accuracy_score(all_labels, all_preds)
precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='weighted')

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")

print("\nClassification Report:")
print(classification_report(all_labels, all_preds))


Accuracy:  0.8977
Precision: 0.9003
Recall:    0.8977
F1 Score:  0.8979

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.93      0.89      4119
           1       0.93      0.87      0.90      4915

    accuracy                           0.90      9034
   macro avg       0.90      0.90      0.90      9034
weighted avg       0.90      0.90      0.90      9034



In [26]:
import torch
torch.cuda.empty_cache()

In [27]:
from torch.utils.data import DataLoader

test_dataloader_im = DataLoader(
    test_dataset_im, batch_size=16, shuffle=False
)

In [28]:
import torch
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

model.eval()
model.to("cuda" if torch.cuda.is_available() else "cpu")

all_preds = []
all_labels = []
i=0
import time

# Start timer
start_time = time.time()
with torch.no_grad():
    for batch in test_dataloader_im:
        input_ids = batch['input_ids'].to(model.device)
        attention_mask = batch['attention_mask'].to(model.device)
        labels = batch['labels'].to(model.device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        preds = torch.argmax(logits, dim=-1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        i+=1
        if i%100==0:
            print(str(i)+" done")
end_time = time.time()
elapsed = end_time - start_time

# Report time
print(f"\n🕒 Evaluation took {elapsed:.2f} seconds ({elapsed/60:.2f} minutes)")


100 done
200 done
300 done
400 done
500 done
600 done
700 done
800 done
900 done
1000 done
1100 done
1200 done
1300 done
1400 done
1500 done
1600 done
1700 done
1800 done
1900 done
2000 done
2100 done
2200 done
2300 done
2400 done
2500 done
2600 done
2700 done
2800 done
2900 done

🕒 Evaluation took 1362.86 seconds (22.71 minutes)


In [29]:
accuracy = accuracy_score(all_labels, all_preds)
precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='weighted')

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")

print("\nClassification Report:")
print(classification_report(all_labels, all_preds))


Accuracy:  0.8890
Precision: 0.8909
Recall:    0.8890
F1 Score:  0.8894

Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.88      0.90     27731
           1       0.85      0.90      0.87     19811

    accuracy                           0.89     47542
   macro avg       0.88      0.89      0.89     47542
weighted avg       0.89      0.89      0.89     47542

